# Usage Notes

This notebook shows practical examples for working with WRF-style NetCDF outputs using **xarray**:

- Calculating **wind speed** from wind components
- Retrieving **wind components at different heights**
- Selecting the **nearest height** (e.g., 50 m AGL) when height levels are available
- Notes on common WRF variable conventions (`U10`, `V10`, `U`, `V`, `PH`, `PHB`, etc.)

> These examples are written to be robust across slightly different datasets. If a variable is missing, the code prints helpful hints.


In [ ]:
# Core imports
from pathlib import Path
import numpy as np
import xarray as xr

# Optional plotting (uncomment if you want quick checks)
# import matplotlib.pyplot as plt

# -------------------------
# Load a dataset
# -------------------------
# Point to a NetCDF file (WRF output or derived dataset).
# Example:
#   ds = xr.open_dataset("/path/to/wrfout_d02_2017-02-06_00:00:00.nc")
#
# If you have multiple files (e.g., time chunks), you can do:
#   ds = xr.open_mfdataset(sorted(Path("/path").glob("wrfout_d02_*.nc")), combine="by_coords")
#
# Set your path here:
DATA_PATH = None  # e.g., "/total/workspace_luan/downscaling/data/wrfout_d02_2017-02-06_00:00:00.nc"

if DATA_PATH is None:
    print("Set DATA_PATH to your NetCDF file path to run the examples.")
else:
    ds = xr.open_dataset(DATA_PATH)
    print(ds)

## Helpers

A few utilities to keep the examples readable:
- A safe variable getter
- A wind-speed calculator
- Convenience functions to guess common wind component variables


In [ ]:
def get_var(ds: xr.Dataset, names):
    """Return the first variable in `names` that exists in ds, else None."""
    for n in names:
        if n in ds.variables:
            return ds[n]
    return None

def wind_speed(u, v):
    """Compute wind speed from components."""
    return np.sqrt(u**2 + v**2)

def guess_surface_uv(ds: xr.Dataset):
    """Try to find 10 m wind components."""
    u10 = get_var(ds, ["U10", "u10", "u_10m", "U_10M"])
    v10 = get_var(ds, ["V10", "v10", "v_10m", "V_10M"])
    return u10, v10

def guess_profile_uv(ds: xr.Dataset):
    """Try to find profile wind components (typically on model levels)."""
    u = get_var(ds, ["U", "u"])
    v = get_var(ds, ["V", "v"])
    return u, v

## 1) Wind speed at 10 m (U10/V10)

If your dataset contains `U10` and `V10` (very common in WRF outputs), computing wind speed is straightforward.


In [ ]:
if 'ds' in globals():
    u10, v10 = guess_surface_uv(ds)
    if u10 is None or v10 is None:
        print("Could not find 10 m components (U10/V10).")
        print("Hints: check ds.data_vars for names containing '10' or 'wind'.")
        print("Example matches:", [v for v in ds.data_vars if ('10' in v.lower() or 'wind' in v.lower())][:40])
    else:
        ws10 = wind_speed(u10, v10).rename("WSPD10")
        ws10.attrs["long_name"] = "10 m wind speed"
        ws10.attrs["units"] = getattr(u10, "units", "")
        print(ws10)
        print("WSPD10 min/max:", float(ws10.min()), float(ws10.max()))

## 2) Wind components at different heights

There are two common situations:

### A) You already have winds on explicit height levels
Some datasets store winds on fixed heights (e.g., 10 m, 50 m, 100 m) with a coordinate like `height`, `z`, `height_agl`, etc.

### B) You have winds on WRF model levels (`U`, `V`) and need to compute height (AGL) from geopotential
In raw WRF output, `U` and `V` are on staggered grids and vertical coordinates are model levels, not meters.
To get height in meters, you typically use geopotential (`PH`, `PHB`) to derive height at mass levels.


### 2A) If your dataset has a height coordinate

This block finds a plausible height coordinate and shows how to select winds at a target height (nearest).

> If you see a coordinate like `height_agl` or `height`, you're in the easy case.


In [ ]:
if 'ds' in globals():
    height_coord = None
    for cand in ["height_agl", "height", "z", "Z", "z_agl", "level_height"]:
        if cand in ds.coords:
            height_coord = cand
            break

    if height_coord is None:
        print("No obvious height coordinate found in ds.coords.")
        print("Coords:", list(ds.coords))
    else:
        print("Using height coordinate:", height_coord)

        # Guess wind components that share this coordinate
        u = get_var(ds, ["u", "U", "u_wind", "U_wind"])
        v = get_var(ds, ["v", "V", "v_wind", "V_wind"])

        if u is None or v is None:
            print("Could not guess profile wind components in this dataset.")
            print("Try specifying the variable names manually.")
        else:
            target_h = 50.0  # meters AGL, change as needed
            u_h = u.sel({height_coord: target_h}, method="nearest")
            v_h = v.sel({height_coord: target_h}, method="nearest")
            ws_h = wind_speed(u_h, v_h).rename(f"WSPD_{int(target_h)}m")
            print(ws_h)

### 2B) Raw WRF output: compute height (AGL) from geopotential (PH/PHB)

WRF stores geopotential at **staggered vertical levels** (`bottom_top_stag`). A typical workflow is:

1. Compute geopotential height at staggered levels: `z_stag = (PH + PHB) / g`
2. Convert to **mass levels** by averaging adjacent staggered levels
3. (Optional) Convert to **AGL** by subtracting terrain height (`HGT`)

Then you can select the nearest model level to a target AGL height (e.g., 50 m).

> Note: `U` and `V` are typically on staggered horizontal grids (`west_east_stag` / `south_north_stag`). For many applications, you want to destagger them first.


In [ ]:
G = 9.81

def destagger(da: xr.DataArray, dim: str) -> xr.DataArray:
    """Destagger a WRF-staggered variable by averaging adjacent points along `dim`."""
    if dim not in da.dims:
        return da
    return 0.5 * (da.isel({dim: slice(0, -1)}) + da.isel({dim: slice(1, None)}))

def compute_height_agl(ds: xr.Dataset) -> xr.DataArray:
    """Compute height AGL at mass points from PH/PHB and subtract terrain height if available."""
    ph  = get_var(ds, ["PH"])
    phb = get_var(ds, ["PHB"])
    if ph is None or phb is None:
        raise KeyError("PH/PHB not found. Cannot compute height from geopotential.")

    z_stag = (ph + phb) / G  # staggered vertical levels (bottom_top_stag)

    # Identify the staggered vertical dim name
    vdim = None
    for d in z_stag.dims:
        if "bottom_top_stag" in d:
            vdim = d
            break
    if vdim is None:
        for d in z_stag.dims:
            if ("stag" in d) and ("bottom" in d):
                vdim = d
                break
    if vdim is None:
        raise ValueError(f"Could not identify staggered vertical dim in PH/PHB dims: {z_stag.dims}")

    # Convert to mass levels
    z_mass = 0.5 * (z_stag.isel({vdim: slice(0, -1)}) + z_stag.isel({vdim: slice(1, None)}))

    # Subtract terrain height if present (AGL)
    hgt = get_var(ds, ["HGT", "hgt"])
    if hgt is not None:
        z_agl = (z_mass - hgt).rename("height_agl")
        z_agl.attrs["units"] = "m"
        z_agl.attrs["long_name"] = "height above ground level (mass points)"
        return z_agl

    z_mass = z_mass.rename("height_msl")
    z_mass.attrs["units"] = "m"
    z_mass.attrs["long_name"] = "height above mean sea level (mass points)"
    return z_mass

In [ ]:
if 'ds' in globals():
    u, v = guess_profile_uv(ds)
    if u is None or v is None:
        print("Could not find profile winds (U/V).")
        print("Hint: check ds.data_vars for 'U'/'V' or other wind profile variables.")
    else:
        # Destagger to mass grid if needed
        u_mass = destagger(u, "west_east_stag")
        v_mass = destagger(v, "south_north_stag")

        # Compute height (AGL if HGT exists, else MSL)
        try:
            z = compute_height_agl(ds)
        except Exception as e:
            print("Height computation failed:", e)
            z = None

        if z is not None:
            # Identify a shared vertical dim
            shared_vdim = None
            for d in u_mass.dims:
                if (d in z.dims) and ("bottom_top" in d):
                    shared_vdim = d
                    break
            if shared_vdim is None:
                for d in u_mass.dims:
                    if "bottom_top" in d:
                        shared_vdim = d
                        break

            if shared_vdim is None:
                print("Could not identify a vertical dim to index.")
                print("U dims:", u_mass.dims)
                print("Z dims:", z.dims)
            else:
                target_h = 50.0  # meters (AGL if available)
                absdiff = np.abs(z - target_h)
                k = absdiff.argmin(shared_vdim)

                u_50 = u_mass.isel({shared_vdim: k})
                v_50 = v_mass.isel({shared_vdim: k})
                ws_50 = wind_speed(u_50, v_50).rename("WSPD_50m_nearest")
                ws_50.attrs["long_name"] = "Wind speed at nearest model level to 50 m (AGL if available)"
                print(ws_50)

## 3) Saving computed fields

You can attach derived variables (like wind speed) to a new Dataset and save as NetCDF.


In [ ]:
if 'ds' in globals():
    u10, v10 = guess_surface_uv(ds)
    if u10 is not None and v10 is not None:
        ws10 = wind_speed(u10, v10).rename("WSPD10")
        out = xr.Dataset({"WSPD10": ws10})
        # Example output path:
        # out.to_netcdf("windspeed10m.nc")
        print("Prepared Dataset with WSPD10:", out)